# Multi-Layer Perceptron (MLP)
Multi-Layer Perceptron (MLP) is een type feedforward kunstmatig neuraal netwerk dat bestaat uit meerdere lagen van knooppunten, ook wel neuronen genoemd. Elke laag is volledig verbonden met de volgende laag, wat betekent dat elke neuron in een laag verbonden is met elke neuron in de volgende laag. MLP's worden gebruikt voor zowel classificatie- als regressietaken en zijn in staat om complexe patronen in gegevens te leren door middel van niet-lineaire activatiefuncties en backpropagation voor het optimaliseren van gewichten. Ze zijn bijzonder effectief bij het modelleren van niet-lineaire relaties en worden vaak toegepast in diverse domeinen zoals beeldherkenning, spraakherkenning en natuurlijke taalverwerking.

In [3]:
import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf

# Add the parent directory (project root) to Python path
project_root = os.path.abspath('..')  # Go up one level from current notebook
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from sklearn.model_selection import train_test_split
from utils.whole_dataset_tf import create_libs_tensorflow_dataset

In [12]:
train_dataset, val_dataset, test_dataset, label_encoder, dataset_info = create_libs_tensorflow_dataset(data_directory="../data", batch_size=64 apply_baseline_correction=True, max_files=10)

📂 Scanning HDF5 files...
Found 10 HDF5 files to process
🔧 Baseline correction enabled: HYBRID method
   Parameters: {'window_length': 51, 'polyorder': 3, 'als_lam': 10000.0, 'als_p': 0.01}
🔄 Processing HDF5 files...
  Processing file 1/10: 2024-11-05T11-26-41_nr-001_B1_tread_aided_10Hz_280A.h5

✅ Data loading completed!
Total samples: 1,972
Spectrum shape: (8188,)
🔄 Converting to numpy arrays...
Label encoding: {'innerliner': 0, 'sidewall': 1, 'tread': 2}
🔄 Splitting data...
Train set: 1,182 samples
Validation set: 395 samples
Test set: 395 samples
🔄 Creating TensorFlow datasets...

🎯 TensorFlow Dataset created successfully!
   • Total samples: 1,972
   • Features per sample: 8,188
   • Classes: 3 (innerliner, sidewall, tread)
   • Baseline corrected: True
   • Baseline method: hybrid
   • Method parameters: {'window_length': 51, 'polyorder': 3, 'als_lam': 10000.0, 'als_p': 0.01}
   • Steps per epoch: 36


In [14]:
print(dataset_info)

{'total_samples': 1972, 'train_samples': 1182, 'val_samples': 395, 'test_samples': 395, 'n_features': 8188, 'n_classes': 3, 'class_names': ['innerliner', 'sidewall', 'tread'], 'feature_shape': (8188,), 'steps_per_epoch': 36, 'validation_steps': 12, 'baseline_corrected': True, 'baseline_method': 'hybrid', 'baseline_params': {'window_length': 51, 'polyorder': 3, 'als_lam': 10000.0, 'als_p': 0.01}}


In [25]:
#normalise the data
def normalize_spectrum(spectrum):
    min_val = tf.reduce_min(spectrum)
    max_val = tf.reduce_max(spectrum)
    normalized_spectrum = (spectrum - min_val) / (max_val - min_val)
    return normalized_spectrum
train_dataset = train_dataset.map(lambda x, y: (normalize_spectrum(x), y))
val_dataset = val_dataset.map(lambda x, y: (normalize_spectrum(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalize_spectrum(x), y))

In [23]:
from tensorflow import keras
from tensorflow.keras import layers
model = keras.Sequential([
    layers.Dense(16, activation='relu',),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.summary()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

ValueError: This model has not yet been built. Build the model first by calling `build()` or by calling the model on a batch of data.

In [21]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {test_accuracy:.4f}')

ValueError: When providing an infinite dataset, you must specify the number of steps to run (if you did not intend to create an infinite dataset, make sure to not call `repeat()` on the dataset).